# Pashto FastSpeech2: train on the full dataset (Kaggle)

A Kaggle GPU session ends after 12 h, so training runs over several sessions. Each session runs the same
notebook: it continues from the newest checkpoint, saves a checkpoint at the end of **every epoch**, and stops
by itself (with a checkpoint) before `HOURS` is up, so the session's output is always saved.

**Settings (right panel):** Accelerator **GPU T4 x2**, Internet **on** (needs a phone-verified account).

**Session 1:** *Save Version -> Save & Run All (Commit)*. It runs in the background (the browser can be closed).
The first session also aligns and preprocesses all clips (~1-2 h) before training starts.

**Every later session:** *Add Input -> Your Work -> Notebooks ->* this notebook, so its last output (checkpoints,
logs, features) is under `/kaggle/input`; check that the input is the **newest version** (update it in the
Input panel if Kaggle offers a newer one). Then *Save & Run All (Commit)* again.

**Important:** use *Save & Run All*, not an interactive session: only committed runs keep `/kaggle/working`.
Set `HOURS` below the GPU quota you have left this week, otherwise Kaggle stops the run first and its output is lost.

Output (the version's *Output* tab): `output/ckpt/Pashto/<step>.pth.tar` (every 10000th step + the newest 2 epoch
checkpoints, ~350 MB each), `output/log/` (TensorBoard), `runs/full/metrics/` (loss curves, `losses.csv`),
`runs/full/stats.json` + `speakers.json` (needed with a checkpoint for `tts.py` / `colab_inference.ipynb`),
`preprocessed_data/` (the features, so later sessions skip preprocessing).

In [ ]:
HOURS      = 11.5     # this session's budget; training stops (with a checkpoint) ~15 min before it
STEPS      = 300000   # total steps; in practice the time budget is what stops each session
BATCH_SIZE = 16       # lower to 8 on CUDA out-of-memory (30 s clips)

In [ ]:
import os
REPO_DIR = "/kaggle/temp/pashto-fastspeech2"   # outside /kaggle/working: the code doesn't need saving
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/msds25033/pashto-fastspeech2.git {REPO_DIR}
%cd {REPO_DIR}
!git pull -q && git log -1 --format="%h %s"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python kaggle_train.py --hours {HOURS} --steps {STEPS} --batch-size {BATCH_SIZE}